In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import base64, importlib, json, shutil, sys
import numpy as np
import pandas as pd

INPUT, WORK = Path("/kaggle/input"), Path("/kaggle/working")
OUT = WORK / "day13-flow-recovery"
OUT.mkdir(parents=True, exist_ok=True)

# Locate saved simulator products; select the largest available run for each family.
candidates = {"binary": [], "zeeman": []}
for meta_path in INPUT.rglob("simulation_metadata.json"):
    sim_dir = meta_path.parent
    family = sim_dir.name.lower()
    if family not in candidates:
        continue
    required = [meta_path, sim_dir / "parameters.csv", sim_dir / "observations.npy"]
    if not all(path.is_file() for path in required):
        continue
    try:
        meta = json.loads(meta_path.read_text(encoding="utf-8"))
        draws = int(meta.get("parameter_draws_unique", 0))
        if meta.get("family") == family:
            candidates[family].append((draws, sim_dir, meta))
    except Exception as exc:
        print("Skipping unreadable simulation folder:", sim_dir, type(exc).__name__, str(exc)[:120])

SIMULATION_DIRS = {}
for family in ("binary", "zeeman"):
    if not candidates[family]:
        raise FileNotFoundError(f"No saved {family} simulation found. Attach the successful Day 7 Physics output with metadata, parameters.csv, and observations.npy.")
    draws, sim_dir, meta = max(candidates[family], key=lambda item: (item[0], str(item[1])))
    if draws < 100_000:
        raise ValueError(f"Selected {family} data has only {draws:,} unique parameter draws; attach the Day 7 output with 100,000 draws per family.")
    SIMULATION_DIRS[family] = sim_dir
    print(f"[PASS] {family} simulations: {draws:,} unique parameter draws at {sim_dir}")

# Use the revised local training module; attached Kaggle inputs remain read-only.
PATCH_ROOT = WORK / "day13-flow-source"
PACKAGE_DIR = PATCH_ROOT / "cosmic_black_swan"
PACKAGE_DIR.mkdir(parents=True, exist_ok=True)
(PACKAGE_DIR / "__init__.py").write_text("", encoding="utf-8")
FLOW_SOURCE_B64 = "..."
(PACKAGE_DIR / "flows.py").write_bytes(base64.b64decode(FLOW_SOURCE_B64))
sys.path.insert(0, str(PATCH_ROOT))
importlib.invalidate_caches()
from cosmic_black_swan.flows import train_conditional_flow

print("Updated flow module:", PACKAGE_DIR / "flows.py")
print("Output root:", OUT)


In [ ]:
import json, time
import pandas as pd
import torch

if not torch.cuda.is_available():
    raise RuntimeError("GPU preflight failed. In Kaggle Settings, select GPU T4 x2, then rerun this notebook.")
print("Flow device:", torch.cuda.get_device_name(0))

RESULTS = {}
for family in ("binary", "zeeman"):
    print(f"\n===== Early-stopped flow refit: {family} =====", flush=True)
    started = time.time()
    result = train_conditional_flow(
        simulation_dir=SIMULATION_DIRS[family],
        epochs=60,
        batch_size=1024,
        learning_rate=1e-3,
        seed=42,
        eval_sources=1000,
        posterior_samples=512,
        early_stopping_patience=6,
        output_dir=OUT / family,
    )
    RESULTS[family] = result
    print(f"[DONE] {family}: best epoch={result['best_epoch']}, validation NLL={result['best_validation_nll']:.4f}, test groups={result['test_records']:,}, seconds={time.time()-started:.1f}", flush=True)
    print("90% test coverage:", {k: round(v['interval_90_empirical_coverage'], 4) for k, v in result['heldout_interval_coverage'].items()})

print("\nBoth flow refits finished.")


In [ ]:
import json
import matplotlib.pyplot as plt
import pandas as pd

# Compare with the earlier Day 7 coverage from the saved Kaggle run log.
OLD_COVERAGE_90 = {
    "binary": {"period_days": .616, "phase": .593, "depth_primary": .702, "depth_ratio": .678,
               "duration_fraction": .714, "ellipsoidal_amplitude": .597, "reflection_amplitude": .715, "noise_sigma": .679},
    "zeeman": {"field_gauss": .750, "radial_velocity_km_s": .796, "line_depth": .779,
               "line_sigma_angstrom": .772, "noise_sigma": .810},
}
def wilson_interval(phat, n, z=1.96):
    denominator = 1.0 + z*z/n
    center = (phat + z*z/(2*n)) / denominator
    half = z * ((phat*(1-phat)/n + z*z/(4*n*n)) ** 0.5) / denominator
    return max(0.0, center-half), min(1.0, center+half)

rows = []
for family, result in RESULTS.items():
    for parameter, coverage in result["heldout_interval_coverage"].items():
        previous = OLD_COVERAGE_90[family][parameter]
        new_coverage = coverage["interval_90_empirical_coverage"]
        ci_low, ci_high = wilson_interval(new_coverage, result["eval_parameter_draws"])
        rows.append({
            "family": family,
            "parameter": parameter,
            "previous_day7_90pct_coverage": previous,
            "new_untouched_test_90pct_coverage": new_coverage,
            "new_90pct_coverage_wilson95_low": ci_low,
            "new_90pct_coverage_wilson95_high": ci_high,
            "change": new_coverage - previous,
            "new_50pct_coverage": coverage["interval_50_empirical_coverage"],
            "median_90_width_fraction_of_prior": coverage["median_90_width_fraction_of_prior_range"],
        })
coverage_df = pd.DataFrame(rows)
coverage_df.to_csv(OUT / "day13_flow_coverage_comparison.csv", index=False)
summary = {
    "created_utc": pd.Timestamp.now(tz="UTC").isoformat(),
    "validation_design": "80/10/10 grouped by parameter_id; early stopping on validation NLL; interval coverage on untouched test parameter draws",
    "prior_day7_coverage_reference": OLD_COVERAGE_90,
    "flow_results": {
        family: {
            "parameter_draws_unique": result["parameter_draws_unique"],
            "training_records": result["training_records"],
            "validation_records": result["validation_records"],
            "test_records": result["test_records"],
            "best_epoch": result["best_epoch"],
            "best_validation_nll": result["best_validation_nll"],
            "epochs_run": result["epochs_run"],
            "heldout_interval_coverage": result["heldout_interval_coverage"],
            "model_file": result["model_file"],
        }
        for family, result in RESULTS.items()
    },
    "outputs": ["day13_flow_coverage_comparison.csv", "day13_flow_summary.json", "day13_flow_report.md", "day13_flow_nll_curves.png"],
    "interpretation_limit": "This is synthetic prior-predictive calibration under the simulator. It does not establish real-star posterior validity.",
}
(OUT / "day13_flow_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
report = [
    "# Day 13 conditional flow recovery", "",
    "Training used saved 100K-draw simulations; no simulations were regenerated.",
    "Parameter IDs were separated into 80% training, 10% validation and 10% untouched test groups.",
    "Early stopping selected the checkpoint using validation negative log likelihood.",
    "Coverage was then measured on the untouched test groups.", "",
    coverage_df.to_string(index=False), "",
    "Interpretation: compare new 90% test coverage with 0.90 and the earlier Day 7 value. If it remains materially below 0.90, record the flow as undercovered and do not claim calibrated posterior intervals.",
    "This synthetic test does not establish real-star posterior validity.",
]
(OUT / "day13_flow_report.md").write_text("\n".join(report) + "\n", encoding="utf-8")

fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
for ax, family in zip(axes, ("binary", "zeeman")):
    history = RESULTS[family]["training_history"]
    ax.plot([r["epoch"] for r in history], [r["train_nll"] for r in history], label="train NLL")
    ax.plot([r["epoch"] for r in history], [r["validation_nll"] for r in history], label="validation NLL")
    ax.axvline(RESULTS[family]["best_epoch"], color="black", linestyle="--", alpha=.6, label="best epoch")
    ax.set_title(family)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Negative log likelihood")
    ax.legend()
fig.savefig(OUT / "day13_flow_nll_curves.png", dpi=160)
plt.show()
print("Saved Day 13 recovery artifacts to:", OUT)
print(coverage_df.to_string(index=False))
